In [13]:
import cv2
import numpy as np

In [4]:
names = "coco.names"
yolov4Cfg = "yolov4.cfg"
yolov4Weights = "yolov4.weights"

In [5]:
names = "coco.names"
names = open(names, 'r')

namesTemp = []

for line in names:
    line = line.strip()
    namesTemp.append(line)

names = namesTemp
names

['person',
 'bicycle',
 'car',
 'motorbike',
 'aeroplane',
 'bus',
 'train',
 'truck',
 'boat',
 'traffic light',
 'fire hydrant',
 'stop sign',
 'parking meter',
 'bench',
 'bird',
 'cat',
 'dog',
 'horse',
 'sheep',
 'cow',
 'elephant',
 'bear',
 'zebra',
 'giraffe',
 'backpack',
 'umbrella',
 'handbag',
 'tie',
 'suitcase',
 'frisbee',
 'skis',
 'snowboard',
 'sports ball',
 'kite',
 'baseball bat',
 'baseball glove',
 'skateboard',
 'surfboard',
 'tennis racket',
 'bottle',
 'wine glass',
 'cup',
 'fork',
 'knife',
 'spoon',
 'bowl',
 'banana',
 'apple',
 'sandwich',
 'orange',
 'broccoli',
 'carrot',
 'hot dog',
 'pizza',
 'donut',
 'cake',
 'chair',
 'sofa',
 'pottedplant',
 'bed',
 'diningtable',
 'toilet',
 'tvmonitor',
 'laptop',
 'mouse',
 'remote',
 'keyboard',
 'cell phone',
 'microwave',
 'oven',
 'toaster',
 'sink',
 'refrigerator',
 'book',
 'clock',
 'vase',
 'scissors',
 'teddy bear',
 'hair drier',
 'toothbrush']

In [6]:
model = cv2.dnn.readNet(yolov4Weights, yolov4Cfg)
layers = model.getLayerNames()

In [7]:
model.getUnconnectedOutLayers() # indexes of the output layers (prediction layers)
outputLayers = []

for outputLayer in model.getUnconnectedOutLayers():
    outputLayer = layers[outputLayer-1]
    outputLayers.append(outputLayer)

In [35]:
capture = cv2.VideoCapture(0) # index of camera (0 is primary)

while True:
    active, frame = capture.read()
    frameHeight, frameWidth, channel = frame.shape # channel is 3 for rgb

    blob = cv2.dnn.blobFromImage(frame, 0.0039, size=(416, 416), mean=(0, 0, 0), swapRB=True, crop=False) # openCV reads as BGR so we swap B and R
    # ^ preprocessing
    model.setInput(blob)
    output = model.forward(outputLayers)

    classes = []
    confidenceScores = []
    boundingBoxes = []

    for i in output:
        for j in i:
            scores = j[5:]
            highScore = np.argmax(scores)

            

            confidenceScore = scores[highScore]

            if confidenceScore > 0.5:

                boxWidth = int(j[2]*frameWidth)
                boxHeight = int(j[3]*frameHeight)

                posX = int(j[0]*frameWidth) # multiply as values are relative to framesize
                posY = int(j[1]*frameHeight) # x - 0.5x

                posX -= int(boxWidth/2) # getting top left point for cv2 drawing (from centre)
                posY -= int(boxHeight/2)

                boundingBoxes.append([posX, posY, boxWidth, boxHeight])
                classes.append(names[highScore])
                confidenceScores.append(confidenceScore)
    
    bestBoxes = cv2.dnn.NMSBoxes(boundingBoxes, confidenceScores, 0.5, 0.4) # non-maximum suppression > prevents overlapping boxes
    # returns indexes of boxes to keep

    for i in range (len(boundingBoxes)):
        if i in bestBoxes:
            posX, posY, boxWidth, boxHeight = boundingBoxes[i]
            cv2.rectangle(frame, (posX, posY), (posX+boxWidth, posY+boxWidth), (0, 0, 255), 2) # BGR
            cv2.putText(frame, f"{classes[i]} : {confidenceScores[i]:.1f}", (posX, posY), cv2.FONT_HERSHEY_PLAIN, 1, (0, 200, 0), 1)
    cv2.imshow('Video', frame)

    if cv2.waitKey(1) == 32: # 32 is ASCII for space
        break

cv2.destroyAllWindows()